# 03 — Gold: Dashboard & Visualisations

**Tickets:** A-05, A-06  
**Purpose:** Build dashboard tiles — demand heatmap, KPI cards — and add contextual narrative.

---

## Setup

In [0]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import src.constants

importlib.reload(src.constants)

from pyspark.sql import functions as F  # noqa: E402
from src.constants import DAY_NAME_MAP, GOLD_FACT_TABLE  # noqa: E402

# Read persisted Gold fact table
fact_df = spark.read.table(GOLD_FACT_TABLE)
row_count = fact_df.count()

print(f"\u2713 Setup complete")
print(f"  Gold fact table: {GOLD_FACT_TABLE}  ({row_count:,} rows)")

## A-05 — Demand heatmap (zone × hour-of-day)

In [0]:
# A-05 — Demand heatmap: top 15 zones × hour-of-day

# Identify the 15 highest-demand zones
top_zones = (
    fact_df.groupBy("pickup_zone")
    .agg(F.sum("trip_count").alias("total_trips"))
    .orderBy(F.col("total_trips").desc())
    .limit(15)
    .select("pickup_zone")
)

# Pivot: zone (rows) × hour (columns) → trip_count
heatmap_df = (
    fact_df.join(top_zones, on="pickup_zone")
    .groupBy("pickup_zone")
    .pivot("hour_of_day", list(range(24)))
    .agg(F.sum("trip_count"))
    .fillna(0)
    .toPandas()
)


# Clean up floating-point noise in zone labels (e.g. 40.730000000000004)
def _clean_zone(label: str) -> str:
    parts = label.split(",")
    if len(parts) == 2:
        return f"{float(parts[0]):.2f},{float(parts[1]):.2f}"
    return label


heatmap_df["pickup_zone"] = heatmap_df["pickup_zone"].apply(_clean_zone)
heatmap_df = heatmap_df.set_index("pickup_zone")
heatmap_df = heatmap_df.loc[heatmap_df.sum(axis=1).sort_values(ascending=False).index]
heatmap_df.columns = [f"{int(c):02d}:00" for c in heatmap_df.columns]

# Plot
fig, ax = plt.subplots(figsize=(18, 7))
sns.heatmap(
    heatmap_df,
    cmap="YlOrRd",
    fmt=",.0f",
    annot=False,
    linewidths=0.3,
    ax=ax,
    cbar_kws={"label": "Trip count"},
)
ax.set_title("Taxi Demand Heatmap: Top 15 Zones \u00d7 Hour of Day", fontsize=14)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Pickup Zone (lat, lon grid)")
plt.tight_layout()
plt.show()

## A-05 — KPI cards

In [0]:
# A-05 — KPI cards + supporting charts

# ---- Headline KPIs ----
kpis = fact_df.agg(
    F.sum("trip_count").alias("total_trips"),
    F.sum("total_revenue").alias("total_revenue"),
    F.round(F.sum("total_revenue") / F.sum("trip_count"), 2).alias(
        "avg_revenue_per_trip"
    ),
    F.round(
        F.sum(F.col("avg_trip_duration_min") * F.col("trip_count"))
        / F.sum("trip_count"),
        2,
    ).alias("weighted_avg_duration_min"),
).first()

print("\u2550" * 60)
print("  HEADLINE KPIs (composite: Jan 2015 + Jan\u2013Mar 2016)")
print("\u2550" * 60)
print(f"  \U0001f696  Total trips      : {kpis['total_trips']:>14,}")
print(f"  \U0001f4b0  Total revenue    : ${kpis['total_revenue']:>14,.0f}")
print(f"  \U0001f4b5  Avg fare / trip  : ${kpis['avg_revenue_per_trip']:>10,.2f}")
print(f"  \u23f1   Avg duration     : {kpis['weighted_avg_duration_min']:>10.1f} min")
print("\u2550" * 60)

# ---- Revenue by hour of day ----
rev_hour_pd = (
    fact_df.groupBy("hour_of_day")
    .agg(F.sum("total_revenue").alias("total_revenue"))
    .orderBy("hour_of_day")
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    rev_hour_pd["hour_of_day"],
    rev_hour_pd["total_revenue"] / 1e6,
    color="steelblue",
)
axes[0].set_title("Total Revenue by Hour of Day")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Revenue ($M)")
axes[0].set_xticks(range(0, 24, 2))

# ---- Avg trip duration by day of week ----
dur_day_pd = (
    fact_df.groupBy("day_of_week")
    .agg(
        F.round(
            F.sum(F.col("avg_trip_duration_min") * F.col("trip_count"))
            / F.sum("trip_count"),
            2,
        ).alias("avg_duration_min"),
    )
    .orderBy("day_of_week")
    .toPandas()
)
dur_day_pd["day_name"] = dur_day_pd["day_of_week"].map(DAY_NAME_MAP)

axes[1].bar(
    dur_day_pd["day_name"],
    dur_day_pd["avg_duration_min"],
    color="darkorange",
)
axes[1].set_title("Avg Trip Duration by Day of Week")
axes[1].set_xlabel("Day")
axes[1].set_ylabel("Duration (min)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## A-06 — Contextual narrative

**Why this matters:**

**Demand heatmap** — The heatmap reveals which lat/lon grid zones (\~1.1 km cells) generate the most taxi pickups and at which hours. Midtown Manhattan (`40.76, -73.97`) dominates across all hours, with a pronounced evening peak (18:00–21:00 on weekdays). Fleet operators can use this to pre-position vehicles in high-demand corridors before rush hour, reducing idle time and passenger wait.

**Revenue by hour** — Revenue peaks in the early evening (18:00–20:00) and drops sharply after midnight, tracking demand closely. A secondary morning peak (07:00–09:00) reflects the commute. Revenue per trip is relatively stable, meaning the revenue curve is driven primarily by volume, not price variation.

**Duration by day** — Average trip duration is broadly consistent across the week, with a slight uptick on weekends. This suggests congestion (which lengthens trips) is offset by lower weekend traffic. Drivers and dispatchers should expect similar per-trip time commitments regardless of the day.

### Data caveats

| Caveat | Detail |
|--------|--------|
| **Non-contiguous date range** | All metrics are a composite of January 2015 and January–March 2016 (9-month gap). Seasonal and year-over-year trends cannot be inferred. |
| **Grid-binned zones** | Zones are \~0.01\u00b0 lat/lon bins (\~1.1 km), not official TLC taxi zones or neighbourhood names. |
| **Cash tip blindspot** | 34% of trips paid by cash report `tip_amount = 0` (tips not recorded, not absent). Any tip-related metrics are systematically deflated. |